In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 💻 Code Snippet Explainer

## What We're Building

A tool that takes a code snippet in **any language** and generates:
1. **Line-by-line explanation** — What each part does
2. **Overall summary** — What the code achieves
3. **Edge cases** — Potential issues and corner cases
4. **Possible bugs** — Things that could go wrong
5. **Improvement suggestions** — How to make the code better

## Key Concept: Syntax-Aware Formatting

Unlike generic text, code has structure — functions, loops, conditions,
imports. Our prompts are designed to:
- Detect the programming language automatically
- Explain code using proper terminology for that language
- Format explanations to reference specific line numbers
- Identify language-specific gotchas (e.g., Python's mutable default args)

## Stack
- **LangChain** — prompt templates and chains
- **Ollama (qwen3.5:9b)** — local LLM

## Step 1 — Install & Import Dependencies

In [ ]:
# Install if needed (uncomment)
# !pip install langchain langchain-ollama -q

from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
import json

print("All imports successful!")

## Step 2 — Initialize the LLM

In [ ]:
llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0.2,  # Low temp for precise technical analysis
)

print("LLM ready!")

## Step 3 — Build the Code Explainer Chain

We add line numbers to the code automatically, then ask the LLM to
reference them in its explanation.

In [ ]:
def add_line_numbers(code: str) -> str:
    """Add line numbers to code for reference in explanations."""
    lines = code.strip().split("\n")
    numbered = [f"{i+1:3d} | {line}" for i, line in enumerate(lines)]
    return "\n".join(numbered)


# Test it
sample = """def greet(name):
    return f"Hello, {name}!"

print(greet("World"))"""

print(add_line_numbers(sample))

In [ ]:
# Main explainer prompt
explainer_prompt = ChatPromptTemplate.from_template(
    """You are an expert code reviewer and teacher. Analyze the following code snippet
and provide a detailed explanation.

Your explanation should include:

1. **Language Detection**: Identify the programming language
2. **Purpose**: What does this code do? (2-3 sentences)
3. **Line-by-Line Explanation**: Explain what each significant line/block does.
   Reference line numbers (e.g., "Line 3-5: ...")
4. **Edge Cases**: What inputs or scenarios could cause unexpected behavior?
5. **Possible Bugs**: Any bugs, anti-patterns, or potential issues
6. **Improvements**: How could this code be made better?

Use clear, educational language. Assume the reader is a beginner-to-intermediate
developer who understands basic programming but may not know advanced patterns.

Code (with line numbers):
```
{numbered_code}
```

Detailed explanation:"""
)

explainer_chain = explainer_prompt | llm | StrOutputParser()

print("Code explainer chain ready!")

## Step 4 — Explain Some Code!

Let's test with code snippets of varying complexity.

In [ ]:
def explain_code(code: str) -> str:
    """Explain a code snippet."""
    numbered = add_line_numbers(code)
    print("📋 Code:")
    print(numbered)
    print("\n" + "=" * 60)
    print("🔍 Explanation:\n")
    
    result = explainer_chain.invoke({"numbered_code": numbered})
    print(result)
    print("=" * 60)
    return result

In [ ]:
# Example 1: Python — List comprehension with a subtle bug
python_code = """def get_even_squares(numbers):
    result = [x**2 for x in numbers if x % 2 == 0]
    return result

data = [1, 2, 3, 4, 5, "6", None, 8]
print(get_even_squares(data))"""

_ = explain_code(python_code)

In [ ]:
# Example 2: JavaScript — Async/Await with error handling
js_code = """async function fetchUserData(userId) {
    const response = await fetch(`/api/users/${userId}`);
    const data = await response.json();
    
    localStorage.setItem('user', JSON.stringify(data));
    
    return {
        name: data.name,
        email: data.email,
        isAdmin: data.role === 'admin'
    };
}

fetchUserData(42).then(user => console.log(user));"""

_ = explain_code(js_code)

In [ ]:
# Example 3: Python — Decorator pattern
decorator_code = """import time
import functools

def retry(max_attempts=3, delay=1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f"Attempt {attempt} failed: {e}. Retrying...")
                    time.sleep(delay * attempt)
        return wrapper
    return decorator

@retry(max_attempts=3, delay=2)
def fetch_data(url):
    import requests
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    return response.json()"""

_ = explain_code(decorator_code)

## Step 5 — Structured Bug Analysis

Let's create a specialized chain that outputs structured bug reports.

In [ ]:
class BugReport(BaseModel):
    """A potential bug found in the code."""
    severity: str = Field(description="critical, warning, or info")
    line_numbers: str = Field(description="Affected line numbers")
    description: str = Field(description="What the bug is")
    impact: str = Field(description="What could go wrong")
    fix: str = Field(description="How to fix it (code suggestion)")


class CodeAnalysis(BaseModel):
    """Complete code analysis result."""
    language: str = Field(description="Programming language detected")
    purpose: str = Field(description="What the code does")
    bugs: list[BugReport] = Field(description="List of potential bugs")
    quality_score: int = Field(description="Code quality score from 1-10")


analysis_parser = JsonOutputParser(pydantic_object=CodeAnalysis)

analysis_prompt = ChatPromptTemplate.from_template(
    """You are a senior code reviewer. Analyze this code for bugs, issues, and quality.

Be thorough:
- Check for security issues (injection, XSS, etc.)
- Check for logic errors
- Check for resource leaks
- Check for race conditions
- Check for edge cases
- Check for performance issues

{format_instructions}

Code:
```
{numbered_code}
```

Analysis as JSON:"""
)

analysis_chain = analysis_prompt | llm | analysis_parser

print("Bug analysis chain ready!")

In [ ]:
# Analyze the JavaScript code for bugs
numbered = add_line_numbers(js_code)

print("🔎 Analyzing code for bugs...\n")
analysis = analysis_chain.invoke({
    "numbered_code": numbered,
    "format_instructions": analysis_parser.get_format_instructions(),
})

print(f"Language: {analysis.get('language')}")
print(f"Purpose: {analysis.get('purpose')}")
print(f"Quality: {analysis.get('quality_score')}/10")
print(f"\n🐛 Bugs Found ({len(analysis.get('bugs', []))}):\n")

for bug in analysis.get("bugs", []):
    severity = bug.get('severity', 'info').upper()
    icon = '🔴' if severity == 'CRITICAL' else '🟡' if severity == 'WARNING' else 'ℹ️'
    print(f"{icon} [{severity}] Lines {bug.get('line_numbers', '?')}")
    print(f"   {bug.get('description', '')}")
    print(f"   Impact: {bug.get('impact', '')}")
    print(f"   Fix: {bug.get('fix', '')}")
    print()

## Step 6 — Code Translation

Bonus feature: translate code between programming languages.

In [ ]:
translate_prompt = ChatPromptTemplate.from_template(
    """You are a polyglot programmer. Translate this code from its current language
to {target_language}.

RULES:
- Use idiomatic {target_language} patterns and conventions
- Preserve the logic and behavior exactly
- Add brief comments explaining {target_language}-specific choices
- Use standard library where possible

Original code:
```
{code}
```

Translated {target_language} code:"""
)

translate_chain = translate_prompt | llm | StrOutputParser()

# Translate Python to Rust
print("🔄 Translating Python → Rust\n")
rust_version = translate_chain.invoke({
    "code": python_code,
    "target_language": "Rust",
})
print(rust_version)

In [ ]:
# Translate JavaScript to Python
print("🔄 Translating JavaScript → Python\n")
python_version = translate_chain.invoke({
    "code": js_code,
    "target_language": "Python (using aiohttp)",
})
print(python_version)

## Step 7 — Complete Code Explainer Pipeline

In [ ]:
def full_code_analysis(code: str, translate_to: str | None = None) -> dict:
    """
    Complete code analysis: explain + bug check + optional translation.
    
    Args:
        code: The code snippet to analyze
        translate_to: Optional target language for translation
    
    Returns:
        Dict with explanation, bugs, and optionally translation
    """
    numbered = add_line_numbers(code)
    result = {}
    
    # 1. Explain
    print("📖 Generating explanation...")
    result["explanation"] = explainer_chain.invoke({"numbered_code": numbered})
    print("   ✓ Done")
    
    # 2. Bug analysis
    print("🐛 Checking for bugs...")
    result["analysis"] = analysis_chain.invoke({
        "numbered_code": numbered,
        "format_instructions": analysis_parser.get_format_instructions(),
    })
    bug_count = len(result["analysis"].get("bugs", []))
    print(f"   ✓ Found {bug_count} potential issue(s)")
    
    # 3. Translation (optional)
    if translate_to:
        print(f"🔄 Translating to {translate_to}...")
        result["translation"] = translate_chain.invoke({
            "code": code,
            "target_language": translate_to,
        })
        print("   ✓ Done")
    
    return result


# Try it!
sql_code = """SELECT u.name, COUNT(o.id) as order_count, SUM(o.total) as revenue
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
WHERE o.created_at >= '2024-01-01'
GROUP BY u.name
HAVING revenue > 1000
ORDER BY revenue DESC
LIMIT 10;"""

result = full_code_analysis(sql_code)
print("\n" + "=" * 60)
print(result["explanation"])

## Step 8 — Try Your Own Code!

In [ ]:
# Paste your own code here!
your_code = """
# Replace this with any code you want to understand
class LinkedList:
    def __init__(self):
        self.head = None
    
    def append(self, data):
        new_node = {"data": data, "next": None}
        if not self.head:
            self.head = new_node
            return
        current = self.head
        while current["next"]:
            current = current["next"]
        current["next"] = new_node
    
    def reverse(self):
        prev = None
        current = self.head
        while current:
            next_node = current["next"]
            current["next"] = prev
            prev = current
            current = next_node
        self.head = prev
"""

explain_code(your_code)

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Line numbering** | Adds references for precise explanations |
| **Language detection** | LLM identifies the programming language |
| **Structured bug reports** | Pydantic models for consistent issue format |
| **Severity levels** | Categorize issues by importance |
| **Code translation** | Convert between languages idiomatically |
| **Multi-stage analysis** | Explain → Bug check → Translate |

## 🔧 Customization Ideas

- **Complexity scoring**: Rate code complexity (cyclomatic, cognitive)
- **Test generation**: Auto-generate unit tests for the code
- **Refactoring**: Suggest and apply refactoring patterns
- **Documentation**: Generate docstrings/JSDoc from code
- **Security audit**: Specialized prompt for OWASP Top 10 checks